# Training CNN-1D URL Classifier & Export ke TFLite
**Google Colab | Dataset 200.000 Balanced | Sequential CNN-1D | Float16 TFLite**

---
Notebook ini melatih model CNN-1D berbasis karakter untuk mengklasifikasikan URL
sebagai **aman** (0) atau **pornografi** (1), kemudian mengekspornya ke format TFLite
yang siap dijalankan di aplikasi Android.

**Arsitektur Sequential CNN-1D (satu lapisan konvolusi):**
```
Input  [1, MAX_LEN]  dtype=INT32  ← array indeks karakter
  │
  └── Embedding(vocab=41, dim=32) → shape [1, MAX_LEN, 32]
  │
  └── Conv1D(num_filters, kernel_size, relu)
  │
  └── MaxPooling1D
  │
  └── Dropout
  │
  └── GlobalMaxPooling1D
  │
  └── Dense(64, relu)
  │
  └── Dense(1, sigmoid)

Output [1, 1]  dtype=FLOAT32  ← probabilitas porno (0.0–1.0)
```

**Alur kerja:**
1. Load dataset → sample 200K balanced
2. Tokenisasi karakter (identik dengan `UrlClassifier.kt`)
3. Domain augmentation (domain penuh + domain inti)
4. Training dengan early stopping (monitor val_loss)
5. Evaluasi (accuracy, F1, confusion matrix, ROC)
6. Export ke TFLite float16
7. Verifikasi TFLite + simpan ke Drive

## Cell 1 — Install Library

In [ ]:
!pip install -q --upgrade scikit-learn seaborn
print('Install selesai.')

## Cell 2 — Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import os
import re
import time
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow.keras import layers, Model
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    f1_score, accuracy_score, precision_score, recall_score,
    confusion_matrix, roc_curve, auc, classification_report
)

print(f'TensorFlow : {tf.__version__}')
gpus = tf.config.list_physical_devices('GPU')
print(f'GPU        : {gpus if gpus else "Tidak tersedia — pastikan Runtime > T4 GPU"}')

# Seed global untuk reprodusibilitas
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

## Cell 3 — Mount Google Drive & Konfigurasi

**Format dataset yang diharapkan** — CSV dengan minimal 2 kolom:
```
url, label
```
atau jika menggunakan dataset RF yang sama:
```
url, label, domain_length, digit_count, ...
```

> `label`: 0 = URL aman, 1 = URL pornografi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ================================================================
# SESUAIKAN PATH INI
# ================================================================
DATASET_PATH    = '/content/drive/MyDrive/Tugas Akhir/Dataset/dataset_100rb.csv'
SAVE_PATH       = '/content/drive/MyDrive/Tugas Akhir/Training/CNN1D/'
BEST_PARAMS_JSON = SAVE_PATH + 'best_params_cnn.json' 
URL_COL         = 'url'
LABEL_COL       = 'label'
N_PER_CLASS     = 50_000  
# ================================================================

os.makedirs(SAVE_PATH, exist_ok=True)

df = pd.read_csv(DATASET_PATH)
print(f'Dataset: {df.shape[0]:,} baris x {df.shape[1]} kolom')
print(f'Kolom  : {list(df.columns)}')
print(f'Distribusi label:\n{df[LABEL_COL].value_counts()}')
df[[URL_COL, LABEL_COL]].head()

## Cell 4 — Fungsi Tokenisasi (IDENTIK dengan Android)

> **KRITIS**: Seluruh kode di cell ini harus 1:1 sama dengan `UrlClassifier.kt`.
> Perbedaan sekecil apapun akan menyebabkan model membaca input yang salah di Android.

### Pemetaan Karakter (CHAR_TO_IDX)
| Range | Indeks | Karakter |
|-------|--------|----------|
| PAD   | 0      | `\u0000` (null / padding) |
| UNK   | 1      | Semua karakter tidak dikenal |
| a–z   | 2–27   | 26 huruf alfabet |
| 0–9   | 28–37  | 10 digit angka |
| .     | 38     | Titik |
| -     | 39     | Tanda hubung |
| _     | 40     | Garis bawah |

**Vocab size = 41** (indeks 0 sampai 40)

In [ ]:
# ============================================================
# CHAR_TO_IDX — identik dengan Android UrlClassifier.kt
# Urutan harus PERSIS sama:
#   put('\u0000', 0)  → chr(0)  = PAD
#   for a..z         → a=2, b=3, ..., z=27
#   for 0..9         → 0=28, 1=29, ..., 9=37
#   '.'=38, '-'=39, '_'=40
# ============================================================
CHAR_TO_IDX: dict = {chr(0): 0}  # PAD
for _i, _c in enumerate('abcdefghijklmnopqrstuvwxyz'):
    CHAR_TO_IDX[_c] = _i + 2       # a=2, ..., z=27
for _i, _c in enumerate('0123456789'):
    CHAR_TO_IDX[_c] = _i + 28      # 0=28, ..., 9=37
CHAR_TO_IDX['.'] = 38
CHAR_TO_IDX['-'] = 39
CHAR_TO_IDX['_'] = 40

PAD_IDX    = 0
UNK_IDX    = 1
VOCAB_SIZE = 41  # 0–40

# TLD yang dikenal — identik dengan COMMON_TLDS di UrlClassifier.kt
COMMON_TLDS = {
    'com','net','org','info','biz','name','pro','int',
    'co','io','me','tv','cc','ws','in','ru','cn','jp','kr',
    'de','uk','fr','it','es','br','au','ca','nl','be','ch',
    'at','pl','se','no','dk','fi','cz','hu','ro','bg','gr',
    'pt','ie','nz','za','sg','hk','tw','my','th','ph','id','vn',
    'xyz','top','site','online','club','live','fun','space',
    'tech','store','shop','app','dev','cloud','digital','media',
    'news','blog','video','games','world','network','global',
    'center','zone','today','one','life','work','money','email',
    'link','click','download','stream','watch','porn','sex',
    'xxx','adult','cam','tube','how',
    'tk','ml','ga','cf','gq',
    'ltd','vip','pw','asia','mobi','tel','travel','jobs','edu','gov','mil'
}

SECOND_LEVEL_TLDS = {
    'co.id','co.uk','co.jp','co.kr','co.nz','co.za','co.in','co.th',
    'com.au','com.br','com.cn','com.hk','com.my','com.sg','com.tw',
    'com.vn','com.ph','com.ar','com.mx','com.co','com.pe','com.ve',
    'com.ec','com.pk','com.bd','com.ng','com.eg','com.tr','com.ua','com.ru',
    'net.au','net.br','net.cn','net.id','net.in','net.nz','net.za',
    'org.au','org.br','org.cn','org.id','org.in','org.nz','org.uk','org.za',
    'ac.id','ac.uk','ac.jp','ac.kr','ac.nz','ac.za','ac.th',
    'edu.au','edu.br','edu.cn','edu.hk','edu.my','edu.sg','edu.tw','edu.vn',
    'go.id','go.jp','go.kr','go.th',
    'or.id','or.jp','or.kr','or.th',
    'ne.jp','ne.kr',
    'web.id','sch.id','my.id','biz.id'
}


def normalize_domain(url: str) -> str:
    """Normalisasi URL → domain — identik dengan Android normalizeDomain()"""
    url = url.lower().strip()
    for prefix in ['https://', 'http://', 'www.']:
        if url.startswith(prefix):
            url = url[len(prefix):]
    url = url.split('/')[0].split('?')[0].split('#')[0].split(':')[0]
    return url


def extract_main_domain_name(full_domain: str) -> str:
    """Ekstrak nama inti domain — identik dengan Android extractMainDomainName()"""
    parts = full_domain.split('.')
    if len(parts) < 2:
        return full_domain
    if len(parts) >= 3:
        potential_2nd = f'{parts[-2]}.{parts[-1]}'
        if potential_2nd in SECOND_LEVEL_TLDS:
            return parts[-3] if len(parts) >= 3 else parts[-2]
    if parts[-1] in COMMON_TLDS:
        return parts[-2]
    return parts[-2] if len(parts) >= 2 else full_domain


def tokenize(domain: str, max_len: int) -> list:
    """Ubah string domain ke array indeks karakter — identik dengan Android tokenize()"""
    tokens = [PAD_IDX] * max_len
    for i, char in enumerate(domain[:max_len]):
        tokens[i] = CHAR_TO_IDX.get(char, UNK_IDX)
    return tokens


# ============================================================
# VERIFIKASI — pastikan output identik dengan Android
# ============================================================
print('Verifikasi pemetaan CHAR_TO_IDX:')
print(f'  a={CHAR_TO_IDX["a"]} (exp=2), z={CHAR_TO_IDX["z"]} (exp=27)')
print(f'  0={CHAR_TO_IDX["0"]} (exp=28), 9={CHAR_TO_IDX["9"]} (exp=37)')
print(f'  .={CHAR_TO_IDX["."]} (exp=38), -={CHAR_TO_IDX["-"]} (exp=39), _={CHAR_TO_IDX["_"]} (exp=40)')
print(f'  VOCAB_SIZE={VOCAB_SIZE} (exp=41)')
print()

print('Verifikasi tokenisasi:')
tests = [
    ('https://www.youporn.com/watch', 'youporn.com', 'youporn'),
    ('http://xvideos.com/video123', 'xvideos.com', 'xvideos'),
    ('https://mail.google.com/inbox', 'mail.google.com', 'google'),
    ('https://pornhub.com', 'pornhub.com', 'pornhub'),
]
print(f'{"URL":<40} {"Domain":<20} {"Core":<15} Tokens[0:8]')
print('-' * 90)
for url, exp_domain, exp_core in tests:
    d = normalize_domain(url)
    core = extract_main_domain_name(d)
    tok = tokenize(core, 34)
    ok_d = '✅' if d == exp_domain else f'❌ (exp:{exp_domain})'
    ok_c = '✅' if core == exp_core else f'❌ (exp:{exp_core})'
    print(f'{url:<40} {d:<20} {core:<15} {tok[:8]}  dom:{ok_d} core:{ok_c}')

## Cell 5 — Sampling 200K Data Balanced

**Dataset yang sama digunakan untuk RF dan CNN-1D** agar perbandingan akademik fair.

Jika `N_PER_CLASS` diubah, ubah juga di `rf_training_v2.ipynb`.

In [ ]:
df_safe = df[df[LABEL_COL] == 0]
df_porn = df[df[LABEL_COL] == 1]

print(f'Total URL aman      : {len(df_safe):,}')
print(f'Total URL pornografi: {len(df_porn):,}')

assert len(df_safe) >= N_PER_CLASS, f'Data aman kurang dari {N_PER_CLASS:,}'
assert len(df_porn) >= N_PER_CLASS, f'Data porno kurang dari {N_PER_CLASS:,}'

df_bal = pd.concat([
    df_safe.sample(n=N_PER_CLASS, random_state=SEED),
    df_porn.sample(n=N_PER_CLASS, random_state=SEED)
]).sample(frac=1, random_state=SEED).reset_index(drop=True)

print(f'\nDataset setelah sampling  : {len(df_bal):,} baris')
print(f'Distribusi               :\n{df_bal[LABEL_COL].value_counts()}')

## Cell 6 — Preprocessing & Domain Augmentation

**MAX_LEN**: dihitung dari persentil 95 panjang domain. Ini memastikan 95% domain
tidak terpotong, dan model tidak terlalu besar untuk domain pendek.

**Domain Augmentation:**
Setiap URL menghasilkan **2 sampel training**:

| URL Input | Domain Penuh | Domain Inti | Alasan |
|-----------|-------------|------------|--------|
| `https://youporn.com/watch` | `youporn.com` | `youporn` | Core domain lebih informatif |
| `https://xvideos.net` | `xvideos.net` | `xvideos` | TLD berbeda tapi porno |
| `https://news.google.com` | `news.google.com` | `google` | Subdomain bisa menyesatkan |

> Di Android saat inference, hanya **domain inti** yang diproses (`extractMainDomainName()`).
> Augmentasi domain penuh membantu robustness model terhadap variasi URL.

In [ ]:
print('Langkah 1: Ekstrak domain dari URL...')
df_bal['full_domain'] = df_bal[URL_COL].apply(
    lambda u: normalize_domain(str(u))
)
df_bal['core_domain'] = df_bal['full_domain'].apply(extract_main_domain_name)

# Tampilkan contoh
print('Contoh ekstraksi domain:')
print(df_bal[['url', 'full_domain', 'core_domain', LABEL_COL]].head(10).to_string(index=False))

# Filter URL tidak valid (tanpa titik)
n_before = len(df_bal)
df_bal = df_bal[
    df_bal['full_domain'].str.contains(r'\.', regex=True) &
    df_bal['full_domain'].str.len().gt(3)
].reset_index(drop=True)
print(f'\nFilter URL tidak valid: {n_before - len(df_bal):,} baris dihapus')
print(f'Sisa: {len(df_bal):,} baris')

# Hitung MAX_LEN dari data nyata
all_lengths = pd.concat([
    df_bal['full_domain'].str.len(),
    df_bal['core_domain'].str.len()
])

MAX_LEN = int(np.percentile(all_lengths, 95))
MAX_LEN = max(MAX_LEN, 20)   # Minimal 20 karakter
MAX_LEN = min(MAX_LEN, 100)  # Maksimal 100 karakter

print(f'\nAnalisis panjang domain (full + core):')
print(f'  Min     : {all_lengths.min()}')
print(f'  Median  : {all_lengths.median():.0f}')
print(f'  P90     : {np.percentile(all_lengths, 90):.0f}')
print(f'  P95     : {np.percentile(all_lengths, 95):.0f}')
print(f'  P99     : {np.percentile(all_lengths, 99):.0f}')
print(f'  Max     : {all_lengths.max()}')
print(f'  MAX_LEN yang dipakai: {MAX_LEN}')

# Visualisasi distribusi panjang
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(all_lengths, bins=60, color='steelblue', alpha=0.7, edgecolor='white')
ax.axvline(MAX_LEN, color='red', linestyle='--', linewidth=2, label=f'MAX_LEN={MAX_LEN} (P95)')
ax.set_xlabel('Panjang Domain (karakter)')
ax.set_ylabel('Frekuensi')
ax.set_title('Distribusi Panjang Domain (Full + Core)')
ax.legend()
plt.tight_layout()
plt.show()

## Cell 7 — Bangun Dataset dengan Augmentasi

Setiap baris menghasilkan 2 sampel training. Total ~400K sampel dari 200K URL.

In [ ]:
print(f'Membangun array token (MAX_LEN={MAX_LEN})...')
print('Ini mungkin memakan ~1–2 menit...')
t0 = time.time()

X_list, y_list, source_list = [], [], []

for _, row in df_bal.iterrows():
    label       = int(row[LABEL_COL])
    full_domain = str(row['full_domain'])
    core_domain = str(row['core_domain'])

    # Sampel 1: domain inti (primary — ini yang dipakai Android saat inference)
    # Contoh: 'youporn.com' → 'youporn'
    if core_domain and len(core_domain) >= 2:
        X_list.append(tokenize(core_domain, MAX_LEN))
        y_list.append(label)
        source_list.append('core')

    # Sampel 2: domain penuh (augmentasi — meningkatkan robustness)
    # Contoh: 'youporn.com'
    if full_domain and full_domain != core_domain and len(full_domain) >= 4:
        X_list.append(tokenize(full_domain, MAX_LEN))
        y_list.append(label)
        source_list.append('full')

X = np.array(X_list, dtype=np.int32)
y = np.array(y_list, dtype=np.float32)

print(f'Selesai dalam {time.time()-t0:.1f} detik')
print(f'\nTotal sampel (setelah augmentasi): {len(X):,}')
print(f'Shape X  : {X.shape}  (sampel × karakter)')
print(f'dtype X  : {X.dtype}')
print(f'Distribusi: aman={int((y==0).sum()):,}, porno={int((y==1).sum()):,}')
print(f'\nBreakdown sumber:')
for src in ['core', 'full']:
    cnt = source_list.count(src)
    print(f'  {src:<6}: {cnt:,} sampel ({cnt/len(X_list)*100:.1f}%)')

# Contoh token
print(f'\nContoh tokenisasi:')
for domain in ['youporn', 'youporn.com', 'google', 'xvideos']:
    tok = tokenize(domain, MAX_LEN)
    print(f'  {domain:<20} → tokens[0:{len(domain)}] = {tok[:len(domain)]}')

## Cell 8 — Split Train / Validation / Test

Pembagian **70% train / 15% val / 15% test**:
- **Train**: untuk mengoptimasi parameter model
- **Val**: untuk early stopping dan pemilihan epoch terbaik
- **Test**: untuk evaluasi final (tidak pernah dilihat model selama training)

In [ ]:
# Split 70/15/15 dengan stratify (jaga proporsi kelas)
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.15, random_state=SEED, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.176, random_state=SEED, stratify=y_temp
    # 0.176 ≈ 15/85 → sisa 15% untuk val, 70% untuk train
)

print('Pembagian dataset:')
print(f'  Train      : {len(X_train):>8,} sampel ({len(X_train)/len(X)*100:.1f}%)')
print(f'  Validation : {len(X_val):>8,} sampel ({len(X_val)/len(X)*100:.1f}%)')
print(f'  Test       : {len(X_test):>8,} sampel ({len(X_test)/len(X)*100:.1f}%)')
print(f'\nDistribusi kelas di setiap split:')
for split_name, y_split in [('Train', y_train), ('Val', y_val), ('Test', y_test)]:
    aman  = int((y_split == 0).sum())
    porno = int((y_split == 1).sum())
    print(f'  {split_name:<10}: aman={aman:,}, porno={porno:,}, rasio={aman/(aman+porno)*100:.1f}%/{porno/(aman+porno)*100:.1f}%')

## Cell 9 — Load Hyperparameter Terbaik

Jika file `best_params_cnn.json` tidak ada (belum menjalankan tuning),
notebook ini menggunakan **default hyperparameter** yang sudah reasonable.

In [ ]:
# Default hyperparameter — digunakan jika belum melakukan tuning
# (sesuai Tabel III.3, nilai tengah dari setiap rentang)
DEFAULT_PARAMS = {
    'num_filters'  : 64,
    'kernel_size'  : 5,
    'dropout_rate' : 0.2,
    'learning_rate': 1e-3,
    'embed_dim'    : 32,   # tetap, tidak dituning
    'dense_units'  : 64,   # tetap, tidak dituning
}

if os.path.exists(BEST_PARAMS_JSON):
    with open(BEST_PARAMS_JSON) as f:
        params = json.load(f)
    print(f'Loaded dari: {BEST_PARAMS_JSON}')
else:
    params = DEFAULT_PARAMS
    print(f'best_params_cnn.json tidak ditemukan.')
    print(f'   Menggunakan default hyperparameter.')
    print(f'   Jalankan cnn1d_hyperparameter_tuning.ipynb untuk tuning.')

# Ambil nilai dengan fallback ke default
NUM_FILTERS   = int(params.get('num_filters',   DEFAULT_PARAMS['num_filters']))
KERNEL_SIZE   = int(params.get('kernel_size',   DEFAULT_PARAMS['kernel_size']))
DROPOUT_RATE  = float(params.get('dropout_rate',  DEFAULT_PARAMS['dropout_rate']))
LEARNING_RATE = float(params.get('learning_rate', DEFAULT_PARAMS['learning_rate']))
EMBED_DIM     = int(params.get('embed_dim',     DEFAULT_PARAMS['embed_dim']))
DENSE_UNITS   = int(params.get('dense_units',   DEFAULT_PARAMS['dense_units']))

# MAX_LEN diambil dari cell sebelumnya (dihitung dari data)
print(f'\nHyperparameter yang digunakan:')
print(f'  num_filters   : {NUM_FILTERS}')
print(f'  kernel_size   : {KERNEL_SIZE}')
print(f'  dropout_rate  : {DROPOUT_RATE}')
print(f'  learning_rate : {LEARNING_RATE}')
print(f'  embed_dim     : {EMBED_DIM}  (fixed)')
print(f'  dense_units   : {DENSE_UNITS}  (fixed)')
print(f'  max_len       : {MAX_LEN}  (dihitung dari data)')
print(f'  vocab_size    : {VOCAB_SIZE}')

## Cell 10 — Bangun Model Sequential CNN-1D

**Detail arsitektur (sesuai laporan Subbab II.1.9):**

1. **Embedding layer**: mengubah indeks karakter (int32) menjadi vektor padat (float32)
   - Input: `[batch, MAX_LEN]` dtype INT32
   - Output: `[batch, MAX_LEN, embed_dim]`

2. **Conv1D**: ekstraksi pola n-gram dari urutan karakter URL
   - `kernel_size=3`: trigram seperti `sex`, `xxx`, `cam`
   - `kernel_size=5`: pentagram seperti `video`, `adult`
   - `kernel_size=7`: heptagram seperti `youporn`, `xvideos`
   - Nilai kernel_size ditentukan dari tuning (default: 5)

3. **MaxPooling1D**: down-sampling, pertahankan fitur dominan

4. **Dropout**: regularisasi, cegah overfitting

5. **GlobalMaxPooling1D**: ambil nilai fitur paling signifikan dari seluruh feature map

6. **Dense (Fully Connected)**: klasifikasi berdasarkan fitur yang diekstrak

7. **Output sigmoid**: probabilitas URL porno [0.0, 1.0]
   - ≥ 0.7: diklasifikasikan sebagai porno (threshold di Android)
   - < 0.7: diklasifikasikan sebagai aman

> **Kenapa INT32 input?** Android menggunakan `buffer.putInt(token)` untuk model
> dengan Embedding layer. TFLite mempertahankan dtype INT32 meski menggunakan
> float16 quantization untuk weights.

In [ ]:
def build_cnn1d_model(
    vocab_size    : int,
    max_len       : int,
    embed_dim     : int,
    num_filters   : int,
    kernel_size   : int,
    dense_units   : int,
    dropout_rate  : float,
    learning_rate : float
) -> tf.keras.Model:
    """
    Sequential CNN-1D untuk klasifikasi URL berbasis karakter.
    Arsitektur: Embedding → Conv1D → MaxPool → Dropout
               → GlobalMaxPool → Dense → Output(Sigmoid)
    Input dtype INT32 agar kompatibel dengan Android (TFLite INT32 → Embedding).
    """
    # Input: array indeks karakter [batch, max_len] dtype=int32
    inputs = tf.keras.Input(shape=(max_len,), dtype='int32', name='input')

    # Embedding: int32 → float32 vektor padat
    x = layers.Embedding(
        input_dim    = vocab_size,
        output_dim   = embed_dim,
        input_length = max_len,
        name         = 'embedding'
    )(inputs)

    # Conv1D: ekstraksi pola n-gram karakter URL
    x = layers.Conv1D(
        filters     = num_filters,
        kernel_size = kernel_size,
        activation  = 'relu',
        padding     = 'same',
        name        = 'conv1d'
    )(x)

    # MaxPooling1D: down-sampling, pertahankan fitur dominan
    x = layers.MaxPooling1D(name='maxpool')(x)

    # Dropout: regularisasi, cegah overfitting
    x = layers.Dropout(dropout_rate, name='dropout')(x)

    # GlobalMaxPooling1D: ambil nilai fitur paling signifikan dari seluruh feature map
    x = layers.GlobalMaxPooling1D(name='global_maxpool')(x)

    # Dense: Fully Connected Layer untuk klasifikasi
    x = layers.Dense(dense_units, activation='relu', name='dense')(x)

    # Output: probabilitas porno, range [0.0, 1.0]
    output = layers.Dense(1, activation='sigmoid', name='output')(x)

    model = Model(inputs=inputs, outputs=output, name='CNN1D_URL_Classifier')

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss='binary_crossentropy',
        metrics=[
            'accuracy',
            tf.keras.metrics.AUC(name='auc'),
            tf.keras.metrics.Precision(name='precision'),
            tf.keras.metrics.Recall(name='recall'),
        ]
    )
    return model


model = build_cnn1d_model(
    vocab_size    = VOCAB_SIZE,
    max_len       = MAX_LEN,
    embed_dim     = EMBED_DIM,
    num_filters   = NUM_FILTERS,
    kernel_size   = KERNEL_SIZE,
    dense_units   = DENSE_UNITS,
    dropout_rate  = DROPOUT_RATE,
    learning_rate = LEARNING_RATE
)

model.summary(line_length=80)
print(f'\nTotal parameter   : {model.count_params():,}')
print(f'Input shape       : {model.input_shape}')
print(f'Input dtype       : {model.input.dtype.name}')
print(f'Output shape      : {model.output_shape}')

## Cell 11 — Callbacks untuk Training

| Callback | Fungsi |
|----------|--------|
| `EarlyStopping` | Hentikan training jika **val_loss** tidak turun 3 epoch berturut-turut |
| `ModelCheckpoint` | Simpan model terbaik berdasarkan **val_accuracy** tertinggi |
| `ReduceLROnPlateau` | Kurangi learning rate jika val_loss stagnan 2 epoch |
| `CSVLogger` | Catat metrik setiap epoch ke file CSV untuk analisis |

> **Kenapa monitor val_loss untuk EarlyStopping?**
> val_loss adalah sinyal paling langsung bahwa model mulai overfitting.
> Ketika val_loss mulai naik meskipun val_accuracy masih stabil, itu
> tanda awal overfitting yang perlu dihentikan.

In [ ]:
CHECKPOINT_PATH = '/content/cnn1d_best.keras'

callbacks = [
    # Hentikan training jika val_loss tidak turun dalam 3 epoch
    # monitor val_loss: sinyal overfitting paling langsung
    # restore_best_weights: setelah berhenti, kembalikan ke epoch terbaik
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=3, mode='min',
        restore_best_weights=True, verbose=1
    ),

    # Simpan model dengan val_accuracy tertinggi (metrik utama evaluasi)
    tf.keras.callbacks.ModelCheckpoint(
        filepath=CHECKPOINT_PATH, monitor='val_accuracy',
        mode='max', save_best_only=True, verbose=1
    ),

    # Kurangi LR jika val_loss tidak turun dalam 2 epoch berturut-turut
    # factor=0.5: LR baru = LR lama × 0.5
    # min_lr: batas bawah LR
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=2,
        min_lr=1e-6, verbose=1
    ),

    # Log metrik ke CSV untuk analisis lanjutan
    tf.keras.callbacks.CSVLogger('/content/training_log.csv'),
]

print(f'Callbacks siap:')
for cb in callbacks:
    print(f'  - {type(cb).__name__}')

## Cell 12 — Training Model

Estimasi waktu:
- GPU T4 Colab: **5–15 menit**
- CPU saja: 30–60 menit

In [ ]:
BATCH_SIZE = 256  # Lebih besar = lebih cepat, tapi butuh lebih banyak RAM GPU
MAX_EPOCHS = 50   # EarlyStopping akan berhenti lebih awal

print(f'Memulai training...')
print(f'  Batch size : {BATCH_SIZE}')
print(f'  Max epochs : {MAX_EPOCHS}')
print(f'  Train      : {len(X_train):,} sampel')
print(f'  Val        : {len(X_val):,} sampel')
print('-' * 60)

t0 = time.time()
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    batch_size=BATCH_SIZE,
    epochs=MAX_EPOCHS,
    callbacks=callbacks,
    verbose=1
)
durasi = time.time() - t0

n_epochs     = len(history.history['loss'])
best_val_acc = max(history.history['val_accuracy'])
best_ep      = history.history['val_accuracy'].index(best_val_acc) + 1

print(f'\nTraining selesai dalam {durasi/60:.1f} menit')
print(f'Total epoch yang dijalankan : {n_epochs}')
print(f'Epoch terbaik               : {best_ep} (val_accuracy={best_val_acc:.4f})')

## Cell 13 — Plot Training History

In [ ]:
hist = history.history
epochs_range = range(1, len(hist['loss']) + 1)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Training History — CNN-1D URL Classifier', fontsize=14, fontweight='bold')

metrics_to_plot = [
    ('loss',      'Loss',     'orange',   'darkorange'),
    ('accuracy',  'Accuracy', 'royalblue','darkblue'),
    ('auc',       'AUC',      'green',    'darkgreen'),
    ('precision', 'Precision','purple',   'darkviolet'),
]

for ax, (metric, title, c_train, c_val) in zip(axes.flat, metrics_to_plot):
    ax.plot(epochs_range, hist[metric],     color=c_train, label='Train',      linewidth=2)
    ax.plot(epochs_range, hist[f'val_{metric}'], color=c_val, label='Validation', linewidth=2, linestyle='--')
    ax.set_title(title)
    ax.set_xlabel('Epoch')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/content/training_history.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot tersimpan: /content/training_history.png')

## Cell 14 — Evaluasi pada Test Set

Test set adalah data yang **tidak pernah dilihat model** selama training maupun validasi.
Ini adalah estimasi performa yang paling jujur (unbiased).

In [ ]:
print('Evaluasi pada Test Set...')

# Prediksi probabilitas
y_pred_prob = model.predict(X_test, batch_size=512, verbose=0).flatten()

# Threshold 0.7 — sama dengan DEFAULT_THRESHOLD di UrlClassifier.kt
THRESHOLD  = 0.7
y_pred     = (y_pred_prob >= THRESHOLD).astype(int)
y_test_int = y_test.astype(int)

acc  = accuracy_score(y_test_int, y_pred)
prec = precision_score(y_test_int, y_pred, zero_division=0)
rec  = recall_score(y_test_int, y_pred, zero_division=0)
f1   = f1_score(y_test_int, y_pred, zero_division=0)

fpr, tpr, _ = roc_curve(y_test_int, y_pred_prob)
roc_auc = auc(fpr, tpr)

print('=' * 55)
print(f'  Threshold  : {THRESHOLD}')
print(f'  Accuracy   : {acc:.4f}  ({acc*100:.2f}%)')
print(f'  Precision  : {prec:.4f}')
print(f'  Recall     : {rec:.4f}')
print(f'  F1-Score   : {f1:.4f}')
print(f'  AUC        : {roc_auc:.4f}')
print('=' * 55)

# Classification Report (text)
report_str = classification_report(
    y_test_int, y_pred,
    target_names=['Aman (0)', 'Pornografi (1)'], digits=4
)
print('\nClassification Report:')
print(report_str)

# Simpan classification report TXT
with open('/content/classification_report.txt', 'w') as f:
    f.write('Classification Report — CNN-1D URL Classifier\n')
    f.write('=' * 50 + '\n\n')
    f.write(f'Threshold  : {THRESHOLD}\n\n')
    f.write(report_str)
    f.write(f'\nAccuracy   : {acc:.4f}\n')
    f.write(f'Precision  : {prec:.4f}\n')
    f.write(f'Recall     : {rec:.4f}\n')
    f.write(f'F1-Score   : {f1:.4f}\n')
    f.write(f'AUC        : {roc_auc:.4f}\n')

# Simpan classification report CSV
pd.DataFrame(
    classification_report(y_test_int, y_pred,
        target_names=['Aman (0)', 'Pornografi (1)'],
        output_dict=True)
).T.to_csv('/content/classification_report.csv')

print('✅ classification_report.txt + .csv tersimpan')

# Bar chart evaluasi 5 metrik
fig, ax = plt.subplots(figsize=(9, 5))
metrics_vals  = [acc, prec, rec, f1, roc_auc]
metrics_names = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC']
colors = ['#2196F3', '#4CAF50', '#FF9800', '#E91E63', '#9C27B0']
bars = ax.bar(metrics_names, metrics_vals, color=colors, alpha=0.85, edgecolor='white')
for b, v in zip(bars, metrics_vals):
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.005,
            f'{v*100:.2f}%', ha='center', fontweight='bold', fontsize=10)
ax.set_ylim(0, 1.15)
ax.set_title('Metrik Evaluasi — CNN-1D URL Classifier', fontweight='bold', fontsize=12)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('/content/metrics_barchart.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ metrics_barchart.png tersimpan')

# Cek overfitting
y_train_prob = model.predict(X_train[:5000], batch_size=512, verbose=0).flatten()
y_train_pred = (y_train_prob >= THRESHOLD).astype(int)
f1_train = f1_score(y_train[:5000].astype(int), y_train_pred, zero_division=0)
print(f'\nCek Overfitting (sample 5K train): F1_train={f1_train:.4f}, F1_test={f1:.4f}, gap={f1_train-f1:.4f}')

## Cell 15 — Confusion Matrix & ROC Curve

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- Confusion Matrix ---
cm = confusion_matrix(y_test_int, y_pred)
tn, fp, fn, tp = cm.ravel()

sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=['Pred: Aman', 'Pred: Porno'],
    yticklabels=['Actual: Aman', 'Actual: Porno'],
    ax=axes[0], linewidths=0.5
)
axes[0].set_title('Confusion Matrix')
axes[0].set_ylabel('Aktual')
axes[0].set_xlabel('Prediksi')

print(f'\nDetail Confusion Matrix:')
print(f'  TP (benar porno)  : {tp:,}')
print(f'  TN (benar aman)   : {tn:,}')
print(f'  FP (false alarm)  : {fp:,}  ← URL aman dianggap porno')
print(f'  FN (lolos filter) : {fn:,}  ← URL porno dianggap aman')

# --- ROC Curve ---
axes[1].plot(fpr, tpr, color='darkorange', lw=2, label=f'AUC = {roc_auc:.4f}')
axes[1].plot([0, 1], [0, 1], 'k--', lw=1, label='Random')
axes[1].axvline(x=fp/(fp+tn) if (fp+tn) > 0 else 0,
                color='red', linestyle=':', alpha=0.6, label=f'Threshold={THRESHOLD}')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title(f'ROC Curve (AUC = {roc_auc:.4f})')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# --- Distribusi Skor ---
axes[2].hist(y_pred_prob[y_test == 0], bins=50, alpha=0.6,
             label='URL Aman', color='green', density=True)
axes[2].hist(y_pred_prob[y_test == 1], bins=50, alpha=0.6,
             label='URL Porno', color='red', density=True)
axes[2].axvline(THRESHOLD, color='black', linestyle='--',
                linewidth=2, label=f'Threshold={THRESHOLD}')
axes[2].set_xlabel('Skor Prediksi (P_porno)')
axes[2].set_ylabel('Densitas')
axes[2].set_title('Distribusi Skor Prediksi')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/content/evaluation_plots.png', dpi=150, bbox_inches='tight')
plt.show()

## Cell 16 — Export ke TFLite (Float16 Quantization)

**Kenapa Float16?**
- Ukuran model ≈ **50% lebih kecil** dari float32
- Kecepatan inference ≈ **2x lebih cepat** di hardware dengan dukungan float16
- Akurasi **hampir identik** (perbedaan < 0.1%)

**Format yang dihasilkan:**
```
Input  : INT32  [1, MAX_LEN]  ← indeks karakter
Output : FLOAT32 [1, 1]       ← probabilitas porno
```

Android `UrlClassifier.kt` mendeteksi INT32 input secara otomatis:
```kotlin
modelInputIsInt = inputTensor?.dataType()?.name == "INT32"
// → menggunakan buffer.putInt(token) bukan buffer.putFloat(token.toFloat())
```

In [ ]:
TFLITE_PATH    = '/content/CNN1D.tflite'
KERAS_H5_PATH  = '/content/cnn1d_model.h5'

# Simpan model Keras (sebagai backup)
model.save(KERAS_H5_PATH)
print(f'Model Keras tersimpan: {KERAS_H5_PATH}')

print('\nMengkonversi ke TFLite...')

# === KONVERSI TFLITE ===
converter = tf.lite.TFLiteConverter.from_keras_model(model)

# Float16 quantization:
# - Mengkuantisasi weights float32 → float16
# - Input INT32 dan Output FLOAT32 TIDAK diubah
# - Mengurangi ukuran file ~50% dengan kehilangan akurasi minimal
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]

try:
    tflite_model = converter.convert()
    print('✅ Konversi dengan float16 quantization berhasil')
except Exception as e:
    print(f'⚠️  Float16 gagal ({e}), mencoba tanpa quantization...')
    converter2 = tf.lite.TFLiteConverter.from_keras_model(model)
    tflite_model = converter2.convert()
    print('✅ Konversi tanpa quantization berhasil')

with open(TFLITE_PATH, 'wb') as f:
    f.write(tflite_model)

size_kb = len(tflite_model) / 1024
print(f'\nFile TFLite: {TFLITE_PATH}')
print(f'Ukuran    : {size_kb:.1f} KB')

## Cell 17 — Verifikasi TFLite Model

Verifikasi bahwa:
1. Input shape dan dtype sesuai dengan yang diharapkan Android
2. Output shape dan dtype sesuai (1 nilai float32)
3. Prediksi TFLite **identik** dengan prediksi model Keras

> Jika ada perbedaan skor > 0.01 antara Keras dan TFLite, ada masalah konversi.

In [ ]:
print('Verifikasi TFLite model...')
print('=' * 55)

# Load TFLite model
interp = tf.lite.Interpreter(model_path=TFLITE_PATH)
interp.allocate_tensors()

in_details  = interp.get_input_details()
out_details = interp.get_output_details()

print('Input tensor:')
for d in in_details:
    print(f'  name   : {d["name"]}')
    print(f'  shape  : {d["shape"]}  ← harus [1, {MAX_LEN}]')
    print(f'  dtype  : {d["dtype"]}  ← harus numpy.int32')

print('\nOutput tensor:')
for d in out_details:
    print(f'  name   : {d["name"]}')
    print(f'  shape  : {d["shape"]}  ← harus [1, 1]')
    print(f'  dtype  : {d["dtype"]}  ← harus numpy.float32')

# Validasi
in_dtype  = in_details[0]['dtype']
in_shape  = in_details[0]['shape']
out_dtype = out_details[0]['dtype']
out_shape = out_details[0]['shape']

print('\nValidasi:')
print(f'  Input dtype INT32   : {"✅" if in_dtype == np.int32 else "❌ MASALAH — Android mengharapkan INT32"}')
print(f'  Input len={MAX_LEN}     : {"✅" if in_shape[-1] == MAX_LEN else f"❌ Mismatch: {in_shape[-1]}"}')
print(f'  Output FLOAT32      : {"✅" if out_dtype == np.float32 else "❌ MASALAH"}')
print(f'  Output 1 nilai      : {"✅" if out_shape[-1] == 1 else f"❌ Mismatch: {out_shape[-1]}"}')

print('\n--- Test Prediksi (Sampel Domain) ---')
test_cases = [
    ('youporn',   1, 'PORNO'),
    ('xvideos',   1, 'PORNO'),
    ('pornhub',   1, 'PORNO'),
    ('brazzers',  1, 'PORNO'),
    ('google',    0, 'AMAN'),
    ('youtube',   0, 'AMAN'),
    ('facebook',  0, 'AMAN'),
    ('wikipedia', 0, 'AMAN'),
]

print(f'{"Domain":<15} {"Label":<8} {"Keras":<10} {"TFLite":<10} {"Status"}')
print('-' * 65)

max_diff = 0
for domain, label, label_str in test_cases:
    tokens = np.array([tokenize(domain, MAX_LEN)], dtype=np.int32)
    keras_score  = float(model.predict(tokens, verbose=0)[0][0])
    interp.set_tensor(in_details[0]['index'], tokens)
    interp.invoke()
    tflite_score = float(interp.get_tensor(out_details[0]['index'])[0][0])
    diff   = abs(keras_score - tflite_score)
    max_diff = max(max_diff, diff)
    tflite_pred = 'PORNO' if tflite_score >= THRESHOLD else 'AMAN'
    correct = '✅' if tflite_pred == label_str else '❌'
    print(f'{domain:<15} {label_str:<8} {keras_score:<10.4f} {tflite_score:<10.4f} {correct} diff={diff:.5f}')

print(f'\nMaks. selisih Keras vs TFLite: {max_diff:.5f}', end=' ')
print('✅ OK' if max_diff < 0.01 else '⚠️  Selisih besar, cek konversi!')

# ── Validasi TFLite pada Seluruh Test Set ──
print('\n--- Validasi TFLite pada Seluruh Test Set ---')
tflite_preds_all = []
n_test = len(X_test)
for i in range(n_test):
    if (i + 1) % 10000 == 0 or i == n_test - 1:
        print(f'  Progress: {i+1}/{n_test}', end='\r')
    interp.set_tensor(in_details[0]['index'], X_test[i:i+1])
    interp.invoke()
    tflite_preds_all.append(float(interp.get_tensor(out_details[0]['index'])[0][0]))

tflite_binary = (np.array(tflite_preds_all) >= THRESHOLD).astype(int)
tflite_acc    = accuracy_score(y_test_int, tflite_binary)
tflite_f1     = f1_score(y_test_int, tflite_binary, zero_division=0)

print(f'\nAkurasi Keras  (Test Set) : {acc*100:.4f}%')
print(f'Akurasi TFLite (Test Set) : {tflite_acc*100:.4f}%')
print(f'Selisih                   : {abs(acc - tflite_acc)*100:.4f}%')
print(f'F1     TFLite (Test Set)  : {tflite_f1:.4f}')
print('✅ TFLite konsisten' if abs(acc - tflite_acc) < 0.005 else '⚠️  Selisih > 0.5% — cek konversi')

# ── Simpan model_architecture.txt ──
arch_lines = []
model.summary(print_fn=lambda x: arch_lines.append(x))
with open('/content/model_architecture.txt', 'w') as f:
    f.write('Model Architecture — CNN-1D URL Classifier\n')
    f.write('=' * 55 + '\n\n')
    f.write('\n'.join(arch_lines))
    f.write(f'\n\nTotal Parameters : {model.count_params():,}\n')
    f.write(f'Input dtype      : INT32 [1, {MAX_LEN}]\n')
    f.write(f'Output dtype     : FLOAT32 [1, 1]\n')
    f.write(f'TFLite size      : {len(tflite_model)/1024:.1f} KB (float16)\n')
print('\n✅ model_architecture.txt tersimpan')

## Cell 18 — Simpan ke Google Drive & Ringkasan

File yang perlu dipindahkan ke Android project:
```
app/src/main/assets/CNN1D.tflite
```

Pastikan `ClassifierManager.kt` sudah menggunakan nama file ini:
```kotlin
MLModelType.CNN_1D -> listOf("url_classifier.tflite", "CNN1D.tflite")
```

In [ ]:
import shutil

# ── Simpan semua file ke Google Drive ──
KERAS_SAVE_PATH = SAVE_PATH + 'cnn1d_model.keras'

files_to_save = [
    (TFLITE_PATH,                          SAVE_PATH + 'CNN1D.tflite'),
    (CHECKPOINT_PATH,                      KERAS_SAVE_PATH),
    ('/content/training_history.png',      SAVE_PATH + 'training_history.png'),
    ('/content/evaluation_plots.png',      SAVE_PATH + 'evaluation_plots.png'),
    ('/content/training_log.csv',          SAVE_PATH + 'training_log.csv'),
    ('/content/metrics_barchart.png',      SAVE_PATH + 'metrics_barchart.png'),
    ('/content/classification_report.txt', SAVE_PATH + 'classification_report.txt'),
    ('/content/classification_report.csv', SAVE_PATH + 'classification_report.csv'),
    ('/content/model_architecture.txt',    SAVE_PATH + 'model_architecture.txt'),
]

print('Menyimpan file ke Google Drive...')
for src, dst in files_to_save:
    if os.path.exists(src):
        shutil.copy(src, dst)
        size = os.path.getsize(dst) / 1024
        print(f'  ✅ {os.path.basename(dst):<40} ({size:.1f} KB)')
    else:
        print(f'  ⚠️  {os.path.basename(src)} tidak ditemukan, dilewati')

# ── Simpan evaluation_metrics.json (lengkap) ──
eval_metrics = {
    'model_name'       : 'CNN1D_URL_Classifier',
    'architecture'     : 'Sequential CNN-1D (Embedding→Conv1D→MaxPool→Dropout→GlobalMaxPool→Dense→Output)',
    'hyperparameters'  : {
        'vocab_size'   : VOCAB_SIZE,
        'max_len'      : MAX_LEN,
        'embed_dim'    : EMBED_DIM,
        'num_filters'  : NUM_FILTERS,
        'kernel_size'  : KERNEL_SIZE,
        'dense_units'  : DENSE_UNITS,
        'dropout_rate' : DROPOUT_RATE,
        'learning_rate': LEARNING_RATE,
    },
    'total_params'     : model.count_params(),
    'dataset'          : {
        'total_samples'  : len(X),
        'train_samples'  : len(X_train),
        'val_samples'    : len(X_val),
        'test_samples'   : len(X_test),
        'n_per_class'    : N_PER_CLASS,
        'augmentation'   : 'full_domain + core_domain',
        'split_ratio'    : '70/15/15',
    },
    'training'         : {
        'n_epochs_run'     : len(history.history['loss']),
        'best_epoch'       : best_ep,
        'best_val_accuracy': round(best_val_acc, 4),
        'batch_size'       : BATCH_SIZE,
        'seed'             : SEED,
    },
    'threshold'        : THRESHOLD,
    'evaluation_keras' : {
        'accuracy' : round(acc,     4),
        'precision': round(prec,    4),
        'recall'   : round(rec,     4),
        'f1_score' : round(f1,      4),
        'auc_roc'  : round(roc_auc, 4),
    },
    'evaluation_tflite': {
        'accuracy' : round(tflite_acc, 4),
        'f1_score' : round(tflite_f1,  4),
        'delta_acc': round(abs(acc - tflite_acc), 4),
    },
    'tflite'           : {
        'size_kb'      : round(len(tflite_model) / 1024, 1),
        'quantization' : 'float16',
        'input_dtype'  : str(in_details[0]['dtype']),
        'input_shape'  : list(map(int, in_details[0]['shape'])),
        'output_dtype' : str(out_details[0]['dtype']),
        'output_shape' : list(map(int, out_details[0]['shape'])),
    },
}

json_path = SAVE_PATH + 'evaluation_metrics.json'
with open(json_path, 'w') as f:
    json.dump(eval_metrics, f, indent=2)
print(f'  ✅ {"evaluation_metrics.json":<40} (JSON)')

# ── Ringkasan ──
print(f'\n{"=" * 62}')
print(f'  RINGKASAN PELATIHAN CNN-1D')
print(f'{"=" * 62}')
print(f'  Dataset        : {N_PER_CLASS*2:,} URL')
print(f'  + Augmentasi   : {len(X):,} sampel (core + full domain)')
print(f'  Split          : {len(X_train):,} train / {len(X_val):,} val / {len(X_test):,} test')
print(f'  MAX_LEN        : {MAX_LEN} karakter  |  Params: {model.count_params():,}')
print(f'  TFLite size    : {len(tflite_model)/1024:.1f} KB (float16)')
print(f'  Epoch terbaik  : {best_ep} / {len(history.history["loss"])}  |  val_acc={best_val_acc:.4f}')
print(f'  {"─" * 52}')
print(f'  {"Metrik":<18} {"Keras":>10}  {"TFLite":>10}')
print(f'  {"─" * 42}')
print(f'  {"Accuracy":<18} {acc:>10.4f}  {tflite_acc:>10.4f}')
print(f'  {"F1-Score":<18} {f1:>10.4f}  {tflite_f1:>10.4f}')
print(f'  {"Precision":<18} {prec:>10.4f}  {"—":>10}')
print(f'  {"Recall":<18} {rec:>10.4f}  {"—":>10}')
print(f'  {"AUC-ROC":<18} {roc_auc:>10.4f}  {"—":>10}')
print(f'{"=" * 62}')
print(f'\nSemua file tersimpan di: {SAVE_PATH}')
print(f'\nLangkah berikutnya:')
print(f'  1. Download CNN1D.tflite dari Drive')
print(f'  2. Copy ke: app/src/main/assets/CNN1D.tflite')
print(f'  3. Build & test di Android')

## Cell 19 — Resume Pelatihan (Markdown Report)

In [ ]:
# Hitung nilai sekali agar dipakai di f-string
_total_params  = model.count_params()
_total_samples = len(X)
_n_epochs_run  = len(history.history['loss'])
_tflite_kb     = len(tflite_model) / 1024
_feat_imp_str  = '\n'.join(
    f'  - {col}: {model.feature_importances_[i]:.4f}'
    for i, col in enumerate(FEATURE_COLS)
) if hasattr(model, 'feature_importances_') else ''

resume_md = f"""# Resume Pelatihan CNN-1D URL Classifier

## 1. Informasi Model
- **Jenis Model**: Sequential CNN-1D (Keras/TensorFlow)
- **Arsitektur**: Embedding({VOCAB_SIZE}, {EMBED_DIM}) → Conv1D({NUM_FILTERS}, kernel={KERNEL_SIZE}) → MaxPool → Dropout({DROPOUT_RATE}) → GlobalMaxPool → Dense({DENSE_UNITS}) → Sigmoid
- **Total Parameter**: {_total_params:,}
- **Input**: INT32 [1, {MAX_LEN}] — array indeks karakter (vocab={VOCAB_SIZE})
- **Output**: FLOAT32 [1, 1] — probabilitas URL pornografi

## 2. Dataset
- **Total URL**: {N_PER_CLASS * 2:,} (balanced)
- **Augmentasi**: full_domain + core_domain → {_total_samples:,} sampel
- **Pembagian**: 70% train / 15% validasi / 15% test
- **MAX_LEN**: {MAX_LEN} karakter (persentil 95)

## 3. Training
- **Optimizer**: Adam (lr={LEARNING_RATE})
- **Loss**: Binary Crossentropy
- **Batch Size**: {BATCH_SIZE}
- **Epoch Terbaik**: {best_ep} / {_n_epochs_run}
- **Best Val Accuracy**: {best_val_acc:.4f}

## 4. Hasil Evaluasi (Test Set)
| Metrik | Keras | TFLite |
|--------|-------|--------|
| Accuracy  | {acc:.4f} | {tflite_acc:.4f} |
| F1-Score  | {f1:.4f}  | {tflite_f1:.4f}  |
| Precision | {prec:.4f} | — |
| Recall    | {rec:.4f}  | — |
| AUC-ROC   | {roc_auc:.4f} | — |
| Delta Acc | — | {abs(acc - tflite_acc):.4f} |

## 5. Model TFLite
- **Format**: Float16 Quantization
- **Ukuran**: {_tflite_kb:.1f} KB
- **Input dtype**: INT32 (kompatibel Android)
- **Output dtype**: FLOAT32
- **Threshold**: {THRESHOLD} (sama dengan Android DEFAULT_THRESHOLD)

## 6. File Output
- `CNN1D.tflite` — model untuk Android
- `cnn1d_model.keras` — model Keras (backup)
- `training_history.png` — plot loss/accuracy/auc/precision per epoch
- `evaluation_plots.png` — confusion matrix + ROC curve + distribusi skor
- `metrics_barchart.png` — bar chart 5 metrik evaluasi
- `classification_report.txt` / `.csv` — laporan per kelas
- `model_architecture.txt` — ringkasan arsitektur model
- `training_log.csv` — log metrik per epoch
- `evaluation_metrics.json` — semua metrik dalam format JSON
- `resume_cnn1d.md` — laporan ini
"""

resume_path = SAVE_PATH + 'resume_cnn1d.md'
with open(resume_path, 'w', encoding='utf-8') as f:
    f.write(resume_md)

print(f'✅ Resume tersimpan: {resume_path}')
print()
print(resume_md)